In [ ]:
import torch
import numpy as np
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("datasets/training_dataset.csv")

# Columns: u1, u2, ..., ud, threshold, argmax_induction
feature_cols = [c for c in df.columns if c.startswith("u")] + ["threshold"]
target_col = "optimal_numg0"


In [2]:

y_raw = df[target_col].values.reshape(-1, 1)
y = np.log(y_raw)


In [ ]:

X = df[feature_cols].values
y_raw = df[target_col].values.reshape(-1, 1)

y = np.log(y_raw)

# Train/val split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

class CircuitDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_ds = CircuitDataset(X_train, y_train)
val_ds   = CircuitDataset(X_val, y_val)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False)

class Regressor(nn.Module):
    def __init__(self, input_dim=3, hidden_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
    def forward(self, x):
        return self.net(x)

input_dim = X.shape[1]
model = Regressor(input_dim=input_dim, hidden_dim=32)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(10000):
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(xb)

    val_loss = 0
    model.eval()
    with torch.no_grad():
        for xb, yb in val_loader:
            val_loss += criterion(model(xb), yb).item() * len(xb)

    print(f"Epoch {epoch+1:02d} | "
          f"train_loss={total_loss/len(train_ds):.4f} | "
          f"val_loss={val_loss/len(val_ds):.4f}")


Epoch 01 | train_loss=11.1945 | val_loss=3.5934
Epoch 02 | train_loss=3.1757 | val_loss=2.7136
Epoch 03 | train_loss=2.2611 | val_loss=1.8467
Epoch 04 | train_loss=1.4269 | val_loss=1.0617
Epoch 05 | train_loss=0.7468 | val_loss=0.5167
Epoch 06 | train_loss=0.3845 | val_loss=0.2941
Epoch 07 | train_loss=0.2610 | val_loss=0.2291
Epoch 08 | train_loss=0.2236 | val_loss=0.2070
Epoch 09 | train_loss=0.2058 | val_loss=0.1916
Epoch 10 | train_loss=0.1915 | val_loss=0.1774
Epoch 11 | train_loss=0.1777 | val_loss=0.1650
Epoch 12 | train_loss=0.1646 | val_loss=0.1516
Epoch 13 | train_loss=0.1518 | val_loss=0.1400
Epoch 14 | train_loss=0.1415 | val_loss=0.1356
Epoch 15 | train_loss=0.1331 | val_loss=0.1244
Epoch 16 | train_loss=0.1268 | val_loss=0.1177
Epoch 17 | train_loss=0.1213 | val_loss=0.1157
Epoch 18 | train_loss=0.1176 | val_loss=0.1106
Epoch 19 | train_loss=0.1140 | val_loss=0.1079
Epoch 20 | train_loss=0.1111 | val_loss=0.1071
Epoch 21 | train_loss=0.1085 | val_loss=0.1069


KeyboardInterrupt: 

In [5]:
torch.save(model.state_dict(), "regressor_model.pth")

In [ ]:
idx = np.random.randint(0, len(X_val))

x_sample = X_val[idx:idx+1]      # shape (1, d)
y_true_log = y_val[idx]          # log-transformed target
y_true = y_true_log      # un-transform

x_t = torch.tensor(x_sample, dtype=torch.float32)

with torch.no_grad():
    y_pred_log = model(x_t).numpy().squeeze()
    y_pred = y_pred_log

print("Random validation index:", idx)
print("Input uncertainties + threshold:", x_sample)
print("Actual argmax_induction:", float(y_true))
print("Predicted argmax_induction:", float(y_pred))

Random validation index: 1278
Input uncertainties + threshold: [[0.37034719 0.2277388  0.54001607 0.35      ]]
Actual argmax_induction: 6.7
Predicted argmax_induction: 6.660440921783447


/var/folders/g5/zfzq1xm948s47p2wdqk_5ql00000gn/T/ipykernel_63811/716017705.py:18: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  print("Actual argmax_induction:", float(y_true))


In [4]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from sklearn.model_selection import train_test_split

class Regressor(nn.Module):
    def __init__(self, input_dim=3, hidden_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
    def forward(self, x):
        return self.net(x)

# Initialize the model first (the same architecture as before)
model_load = Regressor(input_dim=4, hidden_dim=32)

# Load the state_dict into the model
model_load.load_state_dict(torch.load("regressor_model.pth"))

model_load.eval()

/var/folders/g5/zfzq1xm948s47p2wdqk_5ql00000gn/T/ipykernel_64488/2586044101.py:22: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_load.load_state_dict(torch.load("regre

Regressor(
  (net): Sequential(
    (0): Linear(in_features=4, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=1, bias=True)
  )
)

In [18]:
xx = [0.521213795535383,
 0.5868067574533484,
 0.8908786980927811, 0.8]
x_t = torch.tensor([xx], dtype=torch.float32)
with torch.no_grad():
    y_pred_log = model_load(x_t).numpy().squeeze()
    y_pred = y_pred_log
print("For input:", xx)
print("Predicted argmax_induction:", float(y_pred))

For input: [0.521213795535383, 0.5868067574533484, 0.8908786980927811, 0.8]
Predicted argmax_induction: 3.7273030281066895
